# BIPV Facade Analysis Pipeline - Colab Runner

This notebook runs the BIPV facade analysis project from GitHub:

https://github.com/DT-GAMER/BIPV_Project

It supports:

- single building image analysis
- multiple image batch analysis
- workflow figure output like the methodology diagram
- BIPV scenario comparison: None, Shadow, Window, Both
- JSON export for downstream reporting

Use a GPU runtime before running:

`Runtime -> Change runtime type -> GPU`

## 1. Mount Google Drive

Your input images and exported results will be read/written from Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or Update the GitHub Project

Run this cell every time you start a fresh Colab runtime. If the project already exists, it pulls the latest version.

In [ ]:
import os

REPO_URL = 'https://github.com/DT-GAMER/BIPV_Project.git'
PROJECT_DIR = '/content/BIPV_Project'

if not os.path.exists(PROJECT_DIR):
    %cd /content
    !git clone {REPO_URL}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
!ls

## 3. Install Dependencies

This project uses SAM, Grounding DINO, LaMa, OpenCV, scikit-image, PyTorch, and related packages.

After installing, Colab may require a runtime restart. If so, restart and continue from Step 4.

In [ ]:
%cd /content/BIPV_Project
!pip install -q -r requirements-colab.txt
print('Dependencies installed. If Colab asks for a restart, restart runtime now.')

## 4. Optional: Fix NumPy Binary Compatibility

Only run this section if you see an error like:

`ValueError: numpy.dtype size changed, may indicate binary incompatibility`

After this cell, restart the runtime.

In [ ]:
# OPTIONAL FIX: run only if NumPy/scikit-image binary compatibility breaks.
# %cd /content/BIPV_Project
# !pip uninstall -y numpy scipy scikit-image opencv-python opencv-python-headless pandas
# !pip install --no-cache-dir --force-reinstall numpy==1.26.4 scipy==1.13.1 scikit-image==0.24.0 opencv-python-headless==4.10.0.84 pandas==2.2.2 matplotlib==3.9.2
# print('Restart runtime after this cell.')

## 5. Import the Project

Run this after installing dependencies or after every runtime restart.

In [ ]:
%cd /content/BIPV_Project

import sys, importlib
sys.path = [p for p in sys.path if p != '/content/BIPV_Project/src']
sys.path.insert(0, '/content/BIPV_Project')
importlib.invalidate_caches()

from src.config import automatic_config, AnalysisConfig
from src.pipeline import run_bipv_analysis
from src.batch import run_batch_analysis
from src.visualization import (
    show_workflow_grid,
    show_bipv_scenario_bars,
    workflow_images_from_result,
)

print('Imports working')

## 6. Configure Input Images

Put your building images in Google Drive. Update the paths below to match your files.

For one image, use `IMAGE_PATH`.

For multiple images, use `IMAGE_PATHS`.

In [ ]:
IMAGE_PATH = '/content/drive/MyDrive/BIPV_Project/IMG_5976.jpeg'

IMAGE_PATHS = [
    '/content/drive/MyDrive/BIPV_Project/IMG_5976.jpeg',
    '/content/drive/MyDrive/BIPV_Project/IMG_6152.jpeg',
    '/content/drive/MyDrive/BIPV_Project/IMG_6133.jpeg',
    '/content/drive/MyDrive/BIPV_Project/IMG_5978.jpeg',
]

OUTPUT_DIR = '/content/drive/MyDrive/BIPV_Project/outputs'
OUTPUT_PATH = '/content/drive/MyDrive/BIPV_Project/pvsyst_export.json'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Single image:', IMAGE_PATH)
print('Batch images:', len(IMAGE_PATHS))
print('Output directory:', OUTPUT_DIR)

## 7A. Run One Image - Automatic Mode

This mode requires no manual dimensions. The model estimates facade dimensions automatically from image/floor/window evidence.

The output area is therefore an **estimated usable BIPV area**, not a measured ground-truth value.

In [ ]:
config = automatic_config(
    image_path=IMAGE_PATH,
    output_path=OUTPUT_PATH,
)

result = run_bipv_analysis(config)

print('Done')
print('Export saved to:', result['output_path'])

## 7B. Optional Calibrated Mode

Use this only when you have measured facade width and height from Google Earth, GIS, architectural drawings, or another reliable source.

Skip this section for fully automatic image-only mode.

In [ ]:
# Optional example. Uncomment and edit values if measured dimensions are available.
# config = AnalysisConfig(
#     image_path=IMAGE_PATH,
#     output_path=OUTPUT_PATH,
#     ge_width_m=42.5,
#     ge_height_m=18.0,
#     require_google_earth_dimensions=True,
# )
# result = run_bipv_analysis(config)

## 8. View One-Image Workflow Result

This shows the five rows used in the methodology figure:

1. Facade Image
2. Obstacle Detection
3. Obstacle Removal
4. Facade Alignment
5. Segmentation Result

In [ ]:
show_workflow_grid(result, column_titles=['Single Building'])

## 9. Print One-Image Engineering Summary

In [ ]:
print('Dimensions:', result['dimensions'])
print('Scaling:', result['stages']['scaling'])
print('Usable area m²:', result['usable_results']['usable_area_m2'])
print('Shadow-reduced usable area m²:', result['usable_results']['usable_area_reduced_m2'])
print('Panel capacity:', result['panel_capacity'])
print('Energy yield:', result['energy_yield'])
print('BIPV scenarios:', result['bipv_scenarios'])

## 10. Plot BIPV Scenario Comparison

These are the paper-style scenarios:

- None: ignores shadows and windows
- Shadow: accounts for shadows only
- Window: accounts for window-to-wall difference only
- Both: accounts for shadows and window-to-wall difference

In [ ]:
show_bipv_scenario_bars(result, metric='annual_kwh')
show_bipv_scenario_bars(result, metric='estimated_kwp')

## 11. Run Multiple Images at Once

Batch mode reuses the loaded models, so it is better than running the notebook from scratch for each image.

In [ ]:
results = run_batch_analysis(
    IMAGE_PATHS,
    output_dir=OUTPUT_DIR,
)

print('Batch complete:', len(results), 'images')

## 12. Display Batch Workflow Grid

In [ ]:
column_titles = ['IMG_5976', 'IMG_6152', 'IMG_6133', 'IMG_5978']
show_workflow_grid(results, column_titles=column_titles)

## 13. Save Workflow Grid as JPG

Use this to create a single image you can send to someone.

In [ ]:
from src.visualization import workflow_images_from_result
import matplotlib.pyplot as plt

stage_sets = [workflow_images_from_result(r) for r in results]
rows = len(stage_sets[0])
cols = len(stage_sets)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 2.3), squeeze=False)

for col, stage_images in enumerate(stage_sets):
    for row, (stage_name, image) in enumerate(stage_images):
        ax = axes[row][col]
        ax.imshow(image)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(stage_name, rotation=0, ha='right', va='center', labelpad=60)
        if row == 0:
            ax.set_title(column_titles[col] if col < len(column_titles) else f'Image {col + 1}')

plt.tight_layout()

SAVE_PATH = '/content/drive/MyDrive/BIPV_Project/workflow_results.jpg'
plt.savefig(SAVE_PATH, dpi=300, bbox_inches='tight')
plt.show()

print('Saved to:', SAVE_PATH)

## 14. Print Batch Summary

In [ ]:
for image_path, r in zip(IMAGE_PATHS, results):
    print('\n---', image_path, '---')
    print('Dimensions:', r['dimensions'])
    print('Usable area m²:', r['usable_results']['usable_area_m2'])
    print('Panel capacity:', r['panel_capacity'])
    print('Annual kWh:', r['energy_yield']['annual_kwh'])
    print('Export:', r['output_path'])